<a href="https://colab.research.google.com/github/Thanjaivalavan/M2_GenAI_AgenticAI/blob/main/05_transformer_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Slides 53-63: Transformer Core Mechanics
------------------------------------------
Implements, from scratch (numpy only)

    Autoregressive factorization:
        P(x1,...,xT) = PRODUCT_t P(x_t | x_1,...,x_{t-1})

    Scaled Dot-Product Attention:
        Attention(Q,K,V) = softmax(Q K^T / sqrt(d_k)) V
        Q = X W_Q,  K = X W_K,  V = X W_V

    Multi-Head Attention:
        head_i = Attention(Q W_Q_i, K W_K_i, V W_V_i)
        MultiHead(Q,K,V) = Concat(head_1,...,head_h) W_O

    Feed-Forward Network:
        FFN(x) = W2 * sigma(W1*x + b1) + b2      (sigma = GELU)

    Output logits -> Softmax with temperature:
        P(x_t=i | x<t) = exp(z_i/T) / sum_j exp(z_j/T)

No PyTorch/TensorFlow — every function below is the literal formula, wired
together into a tiny (untrained, randomly-initialized) single-layer
transformer block so you can see the shapes and the math flow end to end.
"""

import numpy as np

np.random.seed(0)


# ---------------------------------------------------------------------------
# 2. Tokens -> embeddings (slide 55)
# ---------------------------------------------------------------------------
def tokenize_and_embed(text, vocab, embedding_matrix):
    """
    Text -> tokens -> token IDs -> embedding lookup.
    embedding_matrix: shape (|V|, d)
    """
    tokens = text.split()
    ids = [vocab[t] for t in tokens]
    X = embedding_matrix[ids]  # shape (T, d)
    return tokens, ids, X


# ---------------------------------------------------------------------------
# 3A. Scaled Dot-Product Attention (slide 56-57)
# ---------------------------------------------------------------------------
def softmax(z, axis=-1):
    z = z - np.max(z, axis=axis, keepdims=True)  # numerical stability
    e = np.exp(z)
    return e / np.sum(e, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, causal_mask=False):
    """
    Attention(Q,K,V) = softmax(Q K^T / sqrt(d_k)) V
    Q,K,V: shape (T, d_k)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)   # (T, T)

    if causal_mask:
        # token t can only attend to tokens <= t (needed for autoregressive generation)
        T = scores.shape[0]
        mask = np.triu(np.ones((T, T)), k=1).astype(bool)
        scores = np.where(mask, -np.inf, scores)

    weights = softmax(scores, axis=-1)  # (T, T) attention weights, rows sum to 1
    context = weights @ V               # (T, d_k)
    return context, weights


# ---------------------------------------------------------------------------
# 3B. Multi-Head Attention (slide 58)
# ---------------------------------------------------------------------------
class MultiHeadAttention:
    def __init__(self, d_model, n_heads):
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_Q = np.random.randn(d_model, d_model) * 0.1
        self.W_K = np.random.randn(d_model, d_model) * 0.1
        self.W_V = np.random.randn(d_model, d_model) * 0.1
        self.W_O = np.random.randn(d_model, d_model) * 0.1

    def forward(self, X, causal_mask=False):
        T, d_model = X.shape
        Q = X @ self.W_Q
        K = X @ self.W_K
        V = X @ self.W_V

        # split into heads: (T, d_model) -> h x (T, d_k)
        heads_out = []
        for h in range(self.n_heads):
            sl = slice(h * self.d_k, (h + 1) * self.d_k)
            head_context, _ = scaled_dot_product_attention(
                Q[:, sl], K[:, sl], V[:, sl], causal_mask=causal_mask
            )
            heads_out.append(head_context)

        concat = np.concatenate(heads_out, axis=-1)  # (T, d_model)
        return concat @ self.W_O                      # (T, d_model)


# ---------------------------------------------------------------------------
# 3C. Feed-Forward Network with GELU (slide 59)
# ---------------------------------------------------------------------------
def gelu(x):
    """Gaussian Error Linear Unit (tanh approximation)."""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))


class FeedForward:
    def __init__(self, d_model, d_ff):
        self.W1 = np.random.randn(d_model, d_ff) * 0.1
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * 0.1
        self.b2 = np.zeros(d_model)

    def forward(self, x):
        """FFN(x) = W2 . sigma(W1.x + b1) + b2"""
        return gelu(x @ self.W1 + self.b1) @ self.W2 + self.b2


# ---------------------------------------------------------------------------
# 4. Output layer: logits -> softmax with temperature (slide 60-61)
# ---------------------------------------------------------------------------
def logits_to_probs(hidden_state, W_vocab, b_vocab, temperature=1.0):
    """
    z = h^T W_vocab + b                     (raw logits)
    P(x_t=i|x<t) = exp(z_i/T) / sum_j exp(z_j/T)     (temperature-scaled softmax)
    """
    z = hidden_state @ W_vocab + b_vocab
    return softmax(z / temperature)


# ---------------------------------------------------------------------------
# Wire it all together: one transformer block + a tiny "next-token" demo
# ---------------------------------------------------------------------------
def main():
    d_model = 8
    n_heads = 2
    d_ff = 16

    vocab_list = ["I", "love", "AI", "generative", "models"]
    vocab = {w: i for i, w in enumerate(vocab_list)}
    embedding_matrix = np.random.randn(len(vocab_list), d_model) * 0.1

    text = "I love AI"
    tokens, ids, X = tokenize_and_embed(text, vocab, embedding_matrix)
    print(f"Input: \"{text}\"")
    print(f"Tokens: {tokens}  ->  IDs: {ids}")
    print(f"Embeddings shape: {X.shape}  (T={X.shape[0]} tokens, d_model={X.shape[1]})\n")

    # 1. Autoregressive factorization illustration
    print("Autoregressive factorization: P(x1,x2,x3) = P(x1) * P(x2|x1) * P(x3|x1,x2)")
    print(f'  = P("{tokens[0]}") * P("{tokens[1]}"|"{tokens[0]}") '
          f'* P("{tokens[2]}"|"{tokens[0]} {tokens[1]}")\n')

    # 2. Multi-head self-attention (causal, so it mirrors autoregressive generation)
    mha = MultiHeadAttention(d_model, n_heads)
    attn_out = mha.forward(X, causal_mask=True)
    print(f"After Multi-Head Attention (causal): shape {attn_out.shape}")

    # 3. Residual connection (standard transformer block, mentioned on slide 108)
    X_res1 = X + attn_out

    # 4. Feed-forward network
    ffn = FeedForward(d_model, d_ff)
    ffn_out = ffn.forward(X_res1)
    X_res2 = X_res1 + ffn_out
    print(f"After FFN + residual: shape {X_res2.shape}\n")

    # 5. Predict next token from the last position's hidden state
    # (larger init scale than the rest of the network, purely so the demo
    #  logits are spread out enough to visibly show temperature's effect)
    W_vocab = np.random.randn(d_model, len(vocab_list)) * 2.0
    b_vocab = np.zeros(len(vocab_list))
    last_hidden = X_res2[-1]  # hidden state of the final token ("AI")

    print("Next-token distribution at different temperatures:")
    for T in (0.5, 1.0, 2.0):
        probs = logits_to_probs(last_hidden, W_vocab, b_vocab, temperature=T)
        ranked = sorted(zip(vocab_list, probs), key=lambda p: -p[1])
        formatted = ", ".join(f"{w}={p:.3f}" for w, p in ranked)
        label = {0.5: "T<1 (cold, focused)", 1.0: "T=1 (standard)", 2.0: "T>1 (hot, random)"}[T]
        print(f"  {label:20s}: {formatted}")


if __name__ == "__main__":
    main()
